In [1]:
import os

In [2]:
os.chdir('..')

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
api_key = os.getenv("LASTFM_API_KEY")

In [12]:
import requests
import time

def get_top_artists(limit: int = 1000, per_page: int = 200, sleep_time: float = 0.25):
    """
    Fetches the top global artists from Last.fm using the chart.getTopArtists method.
    
    Parameters:
        api_key (str): Your Last.fm API key.
        limit (int): Total number of artists to fetch (default 1000).
        per_page (int): Number of artists per request (max ~200).
        sleep_time (float): Delay between requests in seconds to avoid rate limits.
    
    Returns:
        List[dict]: A list of artist dictionaries with 'name', 'listeners', 'playcount', and 'url'.
    """
    base_url = "https://ws.audioscrobbler.com/2.0/"
    artists = []
    total_pages = (limit + per_page - 1) // per_page  # ceiling division

    for page in range(1, total_pages + 1):
        params = {
            "method": "chart.getTopArtists",
            "api_key": api_key,
            "format": "json",
            "limit": per_page,
            "page": page
        }
        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()

        data = response.json()
        page_artists = data.get("artists", {}).get("artist", [])

        for a in page_artists:
            artists.append({
                "mbid": a.get('mbid'),
                "listeners": int(a.get("listeners", 0)),
                "playcount": int(a.get("playcount", 0)),
            })

        print(f"Fetched page {page} ({len(artists)} total artists)")
        if len(artists) >= limit:
            break

        time.sleep(sleep_time)  # be polite to the API

    return artists[:limit]

In [13]:
top_artists = get_top_artists()

Fetched page 1 (200 total artists)
Fetched page 2 (400 total artists)
Fetched page 3 (600 total artists)
Fetched page 4 (800 total artists)
Fetched page 5 (1000 total artists)


In [14]:
top_artists

[{'mbid': 'c8b03190-306c-4120-bb0b-6f2ebfc06ea9',
  'listeners': 5052236,
  'playcount': 1031561208},
 {'mbid': 'a74b1b7f-71a5-4011-9441-d0b5e4122711',
  'listeners': 7939386,
  'playcount': 1277161366},
 {'mbid': 'f6beac20-5dfe-4d1f-ae02-0b0a740aafd6',
  'listeners': 4094636,
  'playcount': 971230756},
 {'mbid': '20244d07-534f-4eff-b4d4-930878889970',
  'listeners': 5682776,
  'playcount': 3439117290},
 {'mbid': '381086ea-f511-4aba-bdf9-71c753dc5077',
  'listeners': 4855042,
  'playcount': 961699069},
 {'mbid': 'f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387',
  'listeners': 4200169,
  'playcount': 1001926734},
 {'mbid': '650e7db6-b795-4eb5-a702-5ea2fc46c848',
  'listeners': 7437203,
  'playcount': 964308141},
 {'mbid': '1882fe91-cdd9-49c9-9956-8e06a3810bd4',
  'listeners': 2985581,
  'playcount': 547889948},
 {'mbid': None, 'listeners': 7697911, 'playcount': 1426359987},
 {'mbid': '9fff2f8a-21e6-47de-a2b8-7f449929d43f',
  'listeners': 6481859,
  'playcount': 1053024590},
 {'mbid': '63aa26c3-d5

In [15]:
import json
json.dumps(top_artists)

'[{"mbid": "c8b03190-306c-4120-bb0b-6f2ebfc06ea9", "listeners": 5052236, "playcount": 1031561208}, {"mbid": "a74b1b7f-71a5-4011-9441-d0b5e4122711", "listeners": 7939386, "playcount": 1277161366}, {"mbid": "f6beac20-5dfe-4d1f-ae02-0b0a740aafd6", "listeners": 4094636, "playcount": 971230756}, {"mbid": "20244d07-534f-4eff-b4d4-930878889970", "listeners": 5682776, "playcount": 3439117290}, {"mbid": "381086ea-f511-4aba-bdf9-71c753dc5077", "listeners": 4855042, "playcount": 961699069}, {"mbid": "f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387", "listeners": 4200169, "playcount": 1001926734}, {"mbid": "650e7db6-b795-4eb5-a702-5ea2fc46c848", "listeners": 7437203, "playcount": 964308141}, {"mbid": "1882fe91-cdd9-49c9-9956-8e06a3810bd4", "listeners": 2985581, "playcount": 547889948}, {"mbid": null, "listeners": 7697911, "playcount": 1426359987}, {"mbid": "9fff2f8a-21e6-47de-a2b8-7f449929d43f", "listeners": 6481859, "playcount": 1053024590}, {"mbid": "63aa26c3-d59b-4da4-84ac-716b54f1ef4d", "listeners": 3975

In [19]:
os.getcwd()

'c:\\Users\\woodbkb2\\Desktop\\learning\\jhu\\Fall 2025\\606.633 - Social Media Analytics\\project'

In [21]:
with open('./data/top_artists_with_metrics.json', 'w') as f:
    f.write(json.dumps(top_artists))

In [ ]:
import requests
import time

def get_artist_stats_by_mbids(mbids, api_key, sleep_seconds=0.2):
    """
    Given a list of MusicBrainz IDs (mbids) for artists, query Last.fm's artist.getInfo endpoint
    and return a dict mapping mbid -> { 'listeners': int, 'playcount': int, 'name': str }.
    Note: Last.fm does not expose a “followers” count for artists in the public API,
    so this will retrieve available fields (listeners, playcount) only.
    
    Parameters:
        mbids (list of str): list of MBIDs for artists
        api_key (str): your Last.fm API key
        sleep_seconds (float): delay between requests (to avoid hammering the API)
    
    Returns:
        dict: { mbid: { 'name': str, 'listeners': int, 'playcount': int } }
    """
    base_url = "https://ws.audioscrobbler.com/2.0/"
    headers = {
        "User-Agent": "YourAppName/1.0 (contact@example.com)"
    }
    results = {}
    
    for mbid in mbids:
        params = {
            'method': 'artist.getInfo',
            'mbid': mbid,
            'api_key': api_key,
            'format': 'json'
        }
        try:
            resp = requests.get(base_url, params=params, headers=headers)
            resp.raise_for_status()
            data = resp.json()
            
            if 'artist' in data:
                art = data['artist']
                name = art.get('name')
                stats = art.get('stats', {})
                listeners = int(stats.get('listeners', 0))
                playcount = int(stats.get('playcount', 0))
                
                results[mbid] = {
                    'name': name,
                    'listeners': listeners,
                    'playcount': playcount
                }
            else:
                # Some kind of error or no data
                results[mbid] = {
                    'name': None,
                    'listeners': None,
                    'playcount': None
                }
        except Exception as e:
            # On error, just log (or you could raise) and continue
            print(f"Error fetching mbid {mbid}: {e}")
            results[mbid] = {
                'name': None,
                'listeners': None,
                'playcount': None
            }
        
        # Respect rate-limiting / be nice
        time.sleep(sleep_seconds)
    
    return results
